# Move classification NIfTI files by split


In [ ]:
import glob
import os
import shutil as sh
from tqdm import tqdm

DATA_SPLIT = "train"  # options: "train", "val", "test"

SPLIT_CONFIG = {
    "train": {
        "source": r"/path/to/workspace/train_dataset_nifti",
        "target": r"/path/to/workspace/classification_model/train_data",
    },
    "val": {
        "source": r"/path/to/workspace/val_dataset_nifti",
        "target": r"/path/to/workspace/classification_model/val_data",
    },
    "test": {
        "source": r"/path/to/workspace/test_dataset_nifti",
        "target": r"/path/to/workspace/classification_model/test_data",
    },
}

config = SPLIT_CONFIG[DATA_SPLIT]
source_dir = config["source"]
target_dir = config["target"]

nifti_images = glob.glob(os.path.join(source_dir, "*", "*", "*", "*.nii.gz"))
print(DATA_SPLIT, len(nifti_images))


In [ ]:
patients_files = {}
for img_tmp in tqdm(nifti_images):
    parts = img_tmp.split("/")
    subject_path_id = parts[5]
    patients_files.setdefault(subject_path_id, []).append(img_tmp)

print("patients:", len(patients_files))


In [ ]:
for subject_path_id, file_list in patients_files.items():
    transverse_files = []

    for file_path in file_list:
        filename = os.path.basename(file_path)
        if "sag" not in filename and "harmonization" not in filename:
            transverse_files.append(file_path)

    for transverse_file in transverse_files:
        base_path = os.path.dirname(transverse_file)
        harmonize_file = os.path.join(base_path, "harmonization_sample.nii.gz")
        parts = harmonize_file.split("/")
        save_name = parts[5][:4] + "_" + parts[7] + "_" + parts[8]
        save_pth = os.path.join(target_dir, save_name)
        sh.copy(harmonize_file, save_pth)
        print(f"Copied: {harmonize_file} to {save_pth}")
